In [ ]:
import cv2
import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# 1. Helper function to rotate images safely
def rotate_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    # Rotate
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

# 2. Configure Recognizer
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=1,       
    neighbors=8,    
    grid_x=8, 
    grid_y=8
)
def get_data(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    faces = []
    ids = []
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    print("Generating Augmentations (Rotation + Lighting)...")
    
    for img_path in image_paths:
        try:
            img = Image.open(img_path).convert('L') 
            img_np = np.array(img, 'uint8')
            user_id = int(os.path.split(img_path)[-1].split(".")[1])
            
            # Base Pre-processing
            smoothed = cv2.bilateralFilter(img_np, 5, 75, 75)
            enhanced = clahe.apply(smoothed)
            
            # --- AUGMENTATION LIST ---
            images_to_add = [enhanced]
            
            # 1. Geometry: Flip
            images_to_add.append(cv2.flip(enhanced, 1))
            
            # 2. Geometry: Rotate (-10, +10)
            images_to_add.append(rotate_image(enhanced, -10))
            images_to_add.append(rotate_image(enhanced, 10))
            
            # 3. Lighting: Brightness Variations (Crucial for shadows!)
            # Darker version (simulate shadows)
            darker = cv2.convertScaleAbs(enhanced, alpha=1, beta=-40)
            images_to_add.append(darker)
            
            # Brighter version (simulate direct light)
            brighter = cv2.convertScaleAbs(enhanced, alpha=1, beta=40)
            images_to_add.append(brighter)
            
            # Add all variants to dataset
            for image in images_to_add:
                faces.append(image)
                ids.append(user_id)
            
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            
    return faces, ids

# Load Data
data_path = r'C:\Users\hp\Desktop\Attendance-System-Using-Face-Recognition\Dataset\training\Cleaned_Training'
all_faces, all_ids = get_data(data_path)

if len(all_faces) == 0:
    print("Error: No data found.")
    exit()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    all_faces, all_ids, test_size=0.2, random_state=43
)

# Train
print(f"Training on {len(X_train)} images (Augmented with rotation)...")
recognizer.train(X_train, np.array(y_train))

# Save
recognizer.save('trainer.yml')
print("Model saved as trainer.yml")

# Evaluate
correct = 0
total = len(X_test)
print("\n--- Self-Evaluation ---")
for i in range(total):
    predicted_id, confidence = recognizer.predict(X_test[i])
    if predicted_id == y_test[i]:
        correct += 1

print(f"Internal Accuracy: {(correct/total)*100:.2f}%")

Processing images and generating augmentations...
Training on 444 images (Augmented with rotation)...
Model saved as trainer.yml

--- Self-Evaluation ---
Internal Accuracy: 83.93%
